In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from thermal import gr_emis, w_emis_spectral, RprimeCondCyl

In [ ]:
def spectral_net_radiation(T_hot, emissivity_table, VF, A_ps, f_bare, f_active, liner_bool=False):
    """
    Incident radiative power [W] on the powerstick from a hot surface.
    Mirrors PowerStick.incident_energy: integrates spectral_radiance(T_hot) over
    wavelength bands, then multiplies by powerstick area A_ps.

    Parameters
    ----------
    T_hot          : float  Hot surface temperature [K]
    emissivity_table : list  [(wavelength_range, emis_active, emis_bare, emis_inactive), ...]
                             wavelength ranges in microns
    VF             : float  View factor hot->powerstick  (A_ps / A_hot, e.g. D_ps/D_in)
    A_ps           : float  Powerstick surface area [m^2]  (= pi * D_ps * L)
    f_bare         : float  Fraction of powerstick with no PV cell
    f_active       : float  Fraction of non-bare area that is active PV
    liner_bool     : bool   True = tungsten liner (w_emis_spectral), False = graphite (gr_emis)
    """
    h   = 6.62607015e-34
    c   = 2.99792458e8
    k_B = 1.380649e-23

    wavelength_min = emissivity_table[0][0][0]
    wavelength_max = emissivity_table[-1][0][1]
    band_gap       = emissivity_table[0][0][1]
    n_bins = 500

    wavelengths   = np.logspace(np.log10(wavelength_min), np.log10(wavelength_max), n_bins)
    wavelengths_m = wavelengths * 1e-6
    spectral_radiance = np.pi * (2*h*c**2) / wavelengths_m**5 / np.expm1(h*c / (wavelengths_m * k_B * T_hot))

    incident_bins = {tuple(wl_range): 0.0 for wl_range, *_ in emissivity_table}

    for i, wavelength in enumerate(wavelengths):
        if liner_bool:
            emitter_emis = w_emis_spectral(T_hot, wavelength)
        else:
            emitter_emis = gr_emis(T_hot)
        if wavelength < band_gap:
            band_width = wavelengths_m[i+1] - wavelengths_m[i]
        else:
            band_width = wavelengths_m[i] - wavelengths_m[i-1]

        for (wavelength_range, emis_active, emis_bare, emis_inactive) in emissivity_table:
            if wavelength_range[0] <= wavelength < wavelength_range[1]:
                avg_ps_emis = (f_bare * emis_bare
                               + (1 - f_bare) * f_active       * emis_active
                               + (1 - f_bare) * (1 - f_active) * emis_inactive)
                incident_bins[tuple(wavelength_range)] += (
                    spectral_radiance[i] * band_width
                    / (VF * (1 - emitter_emis) / emitter_emis + 1 + (1 - avg_ps_emis) / avg_ps_emis)
                )
                break

    # Absorbed power: mirrors generate_electricity_step — each component fraction sums to avg_ps_emis/avg_ps_emis = 1
    Q_total = sum(incident_bins.values()) * A_ps
    return Q_total


def graphite_to_powerstick_heat_transfer(T_outer, T_ps, k_gr,
                                          emissivity_table, f_bare, f_active,
                                          D_in, D_out, D_ps, L,
                                          liner_bool=False, tol=1e-2, max_iter=50):
    """
    Heat transfer from an annular graphite region to a powerstick via series
    conduction (outer->inner wall) and spectral radiation (inner wall->powerstick).
    Radiation is computed using the same approach as PowerStick.incident_energy /
    generate_electricity_step.

    Parameters
    ----------
    T_outer  : float  Outer graphite wall temperature [K]
    T_ps     : float  Powerstick temperature [K]  (used for brentq lower bound)
    k_gr     : float  Graphite thermal conductivity [W/mK]
    D_in     : float  Inner diameter of graphite annulus [m]
    D_out    : float  Outer diameter of graphite annulus [m]
    D_ps     : float  Powerstick outer diameter [m]
    L        : float  Length [m]
    """
    A_ps     = np.pi * D_ps * L          # powerstick surface area
    VF_gr_ps = D_ps / D_in               # view factor: graphite inner wall -> powerstick
    R_cond   = RprimeCondCyl(D_in, D_out, k_gr) / L

    def residual(T_inner):
        Q_rad = spectral_net_radiation(T_inner, emissivity_table, VF_gr_ps,
                                       A_ps, f_bare, f_active, liner_bool)
        return T_inner - (T_outer - Q_rad * R_cond)

    T_inner = brentq(residual, T_ps + 1e-3, T_outer - 1e-3, xtol=tol, maxiter=max_iter)
    Q = spectral_net_radiation(T_inner, emissivity_table, VF_gr_ps,
                               A_ps, f_bare, f_active, liner_bool)
    return Q, T_inner

In [ ]:
# ── Fixed inputs ──────────────────────────────────────────────────────────────
V_graphite = 10      # total graphite annulus volume [m^3] — fill in
L          = 3      # length [m] — fill in
D_ps       = 0.13      # powerstick outer diameter [m] — fill in
T_ps       = 323      # powerstick temperature [K] — fill in

# Multi-band powerstick emissivity definition
bandgap          = 1.24      # bandgap wavelength [microns], e.g. 1.77 for GaSb
# Below-bandgap emissivities (band 1)
emis_active_1    = 0.95      # active PV cell emissivity below bandgap
emis_bare_1      = 0.05      # bare surface emissivity below bandgap
emis_inactive_1  = 0.2      # inactive cell emissivity below bandgap
# Above-bandgap emissivities (band 2)
emis_active_2    = 0.05      # active PV cell emissivity above bandgap
emis_bare_2      = 0.05      # bare surface emissivity above bandgap
emis_inactive_2  = 0.2      # inactive cell emissivity above bandgap
# Fractional areas
f_bare           = 0.2      # fraction of powerstick with no PV cell
f_active         = 0.85      # fraction of non-bare area that is active PV

emissivity_table = [
    ((1e-1, bandgap), emis_active_1, emis_bare_1, emis_inactive_1),
    ((bandgap, 20), emis_active_2, emis_bare_2, emis_inactive_2),
]

# ── Sweep parameters ──────────────────────────────────────────────────────────
k_gr_vals    = [15, 30]           # graphite thermal conductivity [W/mK]
T_outer_vals = [2273, 2473, 2673] # outer graphite wall temperature [K]
D_in_vals    = [0.4, 1, 1.25]    # inner graphite diameter [m] — adjust units to match above

# ── Sweep ─────────────────────────────────────────────────────────────────────
results = {}
for k_gr in k_gr_vals:
    for D_in in D_in_vals:
        D_out   = np.sqrt(4 * V_graphite / (np.pi * L) + D_in**2)
        Q_vals, T_inner_vals = [], []
        for T_outer in T_outer_vals:
            Q, T_inner = graphite_to_powerstick_heat_transfer(
                T_outer, T_ps, k_gr,
                emissivity_table, f_bare, f_active,
                D_in, D_out, D_ps, L
            )
            Q_vals.append(Q)
            T_inner_vals.append(T_inner)
        results[(k_gr, D_in)] = (Q_vals, T_inner_vals)

# ── Plot ──────────────────────────────────────────────────────────────────────
T_outer_C = np.array(T_outer_vals) - 273.15
colors    = plt.cm.plasma(np.linspace(0.15, 0.85, len(D_in_vals)))
ls_map    = {k: style for k, style in zip(k_gr_vals, ['-', '--'])}

fig_Q, ax_Q = plt.subplots(figsize=(8, 5))
fig_T, ax_T = plt.subplots(figsize=(8, 5))

for ci, D_in in enumerate(D_in_vals):
    D_out = np.sqrt(4 * V_graphite / (np.pi * L) + D_in**2)
    for k_gr in k_gr_vals:
        Q_vals, T_inner_vals = results[(k_gr, D_in)]
        lbl = f"D_in={D_in}, k={k_gr} W/mK"
        ax_Q.plot(T_outer_C, np.array(Q_vals) / 1e3,
                  color=colors[ci], linestyle=ls_map[k_gr], marker='o', label=lbl)
        ax_T.plot(T_outer_C, np.array(T_inner_vals) - 273.15,
                  color=colors[ci], linestyle=ls_map[k_gr], marker='o', label=lbl)

for ax, ylabel, title in [
    (ax_Q, "Q [kW]",       "Power output"),
    (ax_T, "T_inner [°C]", "Graphite inner wall temperature"),
]:
    ax.set_xlabel("T_outer [°C]")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True)
    ax.legend(fontsize=8, ncol=2)

fig_Q.tight_layout()
fig_T.tight_layout()
plt.show()